# 20 — Same sessions through MongoDB
This notebook requires a real MongoDB deployment (local Community Server or Atlas). Installing PyMongo alone does not create a server. It does not require an LLM.

In your Agent1 terminal, install the optional driver:
```bash
python -m pip install "pymongo>=4.6,<5"
```
Set MONGODB_URI in your environment before launching VS Code, or enter it through getpass below. For an already-running local server: `mongodb://localhost:27017/`. For Atlas, use your deployment's connection URI. Do not paste credentials into saved cells or commit them.

This performs real writes of synthetic demo conversations to database `agent1_demo`, collection `agent_checkpoints`. It does not migrate your SQLite records. The adapter was tested here with a test double; this notebook is the live server check.


In [1]:
from pathlib import Path
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open inside Agent1.")
sys.path.insert(0, str(project_root / "src"))
from agent1.session import ChatSession


In [ ]:
# import os

# os.environ["MONGODB_URI"] = "mongodb://localhost:27017/"

In [2]:
from types import SimpleNamespace
import numpy as np
from agent1.analysis.action_plan import ActionPlan
from agent1.persistent_agent import PersistentAgent
from agent1.conversation.context_builder import ContextBuilder
from agent1.routing.action_types import ActionType, RoutingDecision
def factory(store):
    p=ActionPlan.model_validate({'request_type':'multi_action','reason':'fixture','steps':[
        {'intent':'track_order','action':'order_service','arguments':{}},
        {'intent':'availability','action':'product_service','arguments':{'product':'backpack','color':'blue'}}]})
    class Analyzer:
        def analyze(self,**kwargs):return p.model_copy(deep=True)
        def classify_pending_reply(self,*args):return 'abandon'
    routing=RoutingDecision('track_order',1,ActionType.ORDER_SERVICE,1,1,[],False,'fixture')
    return PersistentAgent(
        ensemble_model=SimpleNamespace(classes_=np.array(['track_order']),predict_proba=lambda _:np.array([[1.0]])),
        router=SimpleNamespace(route=lambda *_:routing),analyzer=Analyzer(),
        context_builder=ContextBuilder(),checkpoint_store=store)


In [ ]:
# import os
# from getpass import getpass
# from agent1.persistence.mongo_store import MongoCheckpointStore
# uri = os.environ.get("MONGODB_URI") or getpass("MongoDB URI (hidden): ")
# try:
#     store = MongoCheckpointStore.from_uri(uri, database="agent1_demo")
# except Exception as exc:
#     # Avoid displaying a traceback that could expose connection details.
#     raise RuntimeError("MongoDB connection failed (" + type(exc).__name__ +
#                        "). Check server, URI, credentials, and network access.") from None
# finally:
#     del uri
# print("Connected to MongoDB. No URI printed.")


In [3]:
from getpass import getpass
from urllib.parse import quote_plus

from agent1.persistence.mongo_store import MongoCheckpointStore

# Enter your original password when prompted.
# Python handles URL encoding automatically.
uri = (
    "mongodb+srv://agent1_user:"
    + quote_plus(getpass("MongoDB password: "))
    + "@agent1.lpqwahh.mongodb.net/"
)

try:
    store = MongoCheckpointStore.from_uri(
        uri,
        database="agent1_demo",
    )
    print("Connected to MongoDB. No URI printed.")
except Exception as exc:
    raise RuntimeError(
        f"MongoDB connection failed ({type(exc).__name__}). "
        "Check your database password and Atlas network access."
    ) from None
finally:
    del uri

Connected to MongoDB. No URI printed.


In [4]:
session = ChatSession(store, factory)
conversation_id = session.new()
print("Conversation:", conversation_id)
print(session.send("Track my order and check the blue backpack."))
# No manual save here: the pending turn is already saved.
restored = ChatSession(store, factory)
restored.open(conversation_id)
assert restored.agent.state.get_missing_arguments() == ["0.order_id"]
print(restored.send("ORD-123"))
# Completion is saved automatically too.
reopened = ChatSession(store, factory)
reopened.open(conversation_id)
assert not reopened.agent.state.has_pending_action()
assert sum(m.role.value == "tool" for m in reopened.agent.state.messages) == 2
print("Automatic save / reopen / complete: PASS")
for item in store.list_conversations(limit=5):
    print(item)


Conversation: chat-26792bf6063641eabe3f546c497b2864
For task 1 (track_order), please provide one order number, such as ORD-123.
1. Order ORD-123: shipped. Estimated delivery: 2026-09-23.
2. Mock inventory: blue backpack: 5 available.
Automatic save / reopen / complete: PASS
{'conversation_id': 'chat-26792bf6063641eabe3f546c497b2864', 'created_at': '2026-10-27T17:27:38.135000+00:00', 'updated_at': '2026-10-27T17:27:38.300000+00:00', 'preview': 'Track my order and check the blue backpack.', 'message_count': 6, 'pending': False}


## Inspect the nested document
The logical checkpoint matches SQLite, but MongoDB stores nested fields directly. The order argument keys are represented as name/value entries to avoid dotted dictionary keys.

In [5]:
document = store.collection.find_one({"_id": conversation_id})
print("Top-level fields:", list(document))
print("Runtime:", document["runtime"])
print("Last message:", document["state"]["messages"][-1])


Top-level fields: ['_id', 'checkpoint_version', 'created_at', 'mongo_format_version', 'runtime', 'state', 'updated_at']
Runtime: {'fresh_request': False, 'single_request_text': '', 'multi_request_text': 'Track my order and check the blue backpack.\nFollow-up: ORD-123'}
Last message: {'role': 'assistant', 'content': '1. Order ORD-123: shipped. Estimated delivery: 2026-09-23.\n2. Mock inventory: blue backpack: 5 available.', 'message_type': 'message', 'metadata': {}}


In [6]:
store.close()
print("Client closed; saved conversations remain in MongoDB.")


Client closed; saved conversations remain in MongoDB.


## Comparison
| Aspect | SQLite implementation | MongoDB implementation |
|---|---|---|
| Storage | Local file; one row per conversation | Server; one document per conversation |
| Checkpoint | JSON text in a SQL column | Nested BSON state/runtime fields |
| Save | INSERT … ON CONFLICT UPDATE | update_one(..., upsert=True) |
| List | SELECT … ORDER BY … LIMIT | find().sort().limit() |
| Session code | ChatSession | Same ChatSession |

Both are latest-snapshot stores with last-save-wins behavior; neither is an event log or automatic summary memory. The full conversation is rewritten each save. MongoDB documents have a 16 MiB limit; larger histories need a different layout later. That redesign is outside this small comparison.

Official references: [PyMongo connection](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/connect/mongoclient/), [updates and upsert](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/update/), [MongoDB documents](https://www.mongodb.com/docs/manual/core/document/).
